# Baseline DGA Fault Classification Experiments

This notebook runs the same baseline pipeline defined in src/train_baselines.py and previews the summary metrics.

In [ ]:
from pathlib import Path
import subprocess
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
PROJECT_ROOT

In [ ]:
cmd = [
    str(PROJECT_ROOT / '.venv' / 'Scripts' / 'python.exe'),
    str(PROJECT_ROOT / 'src' / 'train_baselines.py')
]
result = subprocess.run(cmd, cwd=PROJECT_ROOT, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError('Training script failed')

In [ ]:
summary_path = PROJECT_ROOT / 'outputs' / 'metrics' / 'model_summary.csv'
summary_df = pd.read_csv(summary_path)
summary_df

## Maintenance Prioritization Workflow

This section scores the dataset using the saved best model and generates a prioritized maintenance plan with the default cost assumptions.

In [ ]:
cmd = [
    str(PROJECT_ROOT / '.venv' / 'Scripts' / 'python.exe'),
    str(PROJECT_ROOT / 'src' / 'score_maintenance.py')
]
result = subprocess.run(cmd, cwd=PROJECT_ROOT, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError('Maintenance scoring script failed')

In [ ]:
maintenance_summary_path = PROJECT_ROOT / 'outputs' / 'predictions' / 'maintenance_summary.json'
maintenance_plan_path = PROJECT_ROOT / 'outputs' / 'predictions' / 'maintenance_plan.csv'

maintenance_summary = pd.read_json(maintenance_summary_path, typ='series')
maintenance_plan_df = pd.read_csv(maintenance_plan_path)

maintenance_summary, maintenance_plan_df.head(10)

## Hyperparameter Tuning

This section runs randomized-search cross-validation for the baseline model family and writes tuned outputs under outputs/tuning.

In [ ]:
cmd = [
    str(PROJECT_ROOT / '.venv' / 'Scripts' / 'python.exe'),
    str(PROJECT_ROOT / 'src' / 'tune_models.py')
]
result = subprocess.run(cmd, cwd=PROJECT_ROOT, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError('Tuning script failed')

## Threshold And Cost Calibration

This section calibrates maintenance priority thresholds from target workload shares and writes a calibrated plan plus cost sensitivity outputs under outputs/calibration.

In [ ]:
cmd = [
    str(PROJECT_ROOT / '.venv' / 'Scripts' / 'python.exe'),
    str(PROJECT_ROOT / 'src' / 'calibrate_maintenance.py')
]
result = subprocess.run(cmd, cwd=PROJECT_ROOT, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError('Calibration script failed')

In [ ]:
tuned_summary_path = PROJECT_ROOT / 'outputs' / 'tuning' / 'metrics' / 'tuned_model_summary.csv'
calibration_report_path = PROJECT_ROOT / 'outputs' / 'calibration' / 'calibration_report.json'

pd.read_csv(tuned_summary_path), pd.read_json(calibration_report_path, typ='series')